kernel : Python (Pixi)

#### Main Dataset

In [1]:
%load_ext autoreload
%autoreload 2
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

MACNAIR = "macnair"
MACNAIR_VAL = "macnair_validation"
MACNAIR_OLIGO = MACNAIR + "_oligo"
VAL_OLIGO = MACNAIR_VAL + "_oligo"

clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

macnair_oligo = sc.read_h5ad(os.path.join(clustered_h5ad_dir, MACNAIR_OLIGO + ".h5ad"))
val_oligo = sc.read_h5ad(os.path.join(clustered_h5ad_dir, VAL_OLIGO + ".h5ad"))

In [18]:
cell_marker_genes = {
    "OPC": [ "PDGFRA", "SEZ6L", "KCNIP1", "TMEM132C", "NTNG1", "NKAIN3", "CSPG5", "SCN1A", "LAMA4", "PTPRZ1", "LHFPL3", "PCDH15", "NXPH1", ],
    "COP": ["SOX6", "UST", "BRINP1", "SH3RF3", "GPC3", "GPR17", "WASF1", "BCAS1", "EPB41L2", "ENPP6", "RAPGEF1", "EPHB1", "TCF7L2", "PRICKLE1", ],
    "NFOL": ["FRY", "GOLIM4", "DOCK6", "PIP5K1B", "ABLIM2", "RGS16", "PDGFA", "NREP", "GLRA3", "CMTM7", "TMTC1", "MAN1A1", ],
    "MOL1": ["LAMA2", "OPALIN", "SVEP1", "ROR1", "LTBP1", "DYSF", "PLXDC2", ],
    "MOL2": ["SLC5A11", "ACSBG1", "ACTN2", "HIVEP3", "ANKRD18A", "DPYSL5", "LRRC63", "MOBP", "MAG", "MBP", "PDE1A", "RNF220", "NKAIN2", ],
}
plot.plot_dotplot(macnair_oligo, MACNAIR_OLIGO, cell_marker_genes, group = "leiden")

In [5]:
leiden_to_celltype = {
    "0": "MOL",
    "1": "MOL",
    "2": "MOL",
    "3": "MOL",
    "4": "MOL",
    "5": "MOL",
    "6": "MOL",
    "7": "MOL",
    "8": "MOL",
    "9": "MOL",
    "10": "MOL",
    "11": "MOL",
    "12": "MOL",
    "13": "MOL",
    "14": "MOL",
    "15": "MOL",
    "16": "MOL",
    "17": "MOL",
    "18": "MOL",
    "19": "MOL",
    "20": "MOL",
    "21": "MOL",
    "22": "MOL",
    "23": "MOL",
    "24": "MOL",
    "25": "MOL",
    "26": "MOL",
    "27": "MOL",
    "28": "MOL",
    "29": "MOL",
    "30": "MOL",
    "31": "MOL",
    "32": "MOL",
    "33": "MOL",
    "34": "MOL",
    "35": "MOL",
    "36": "MOL",
    "37": "MOL",
    "38": "MOL",
    "39": "MOL",
    "40": "MOL",
    "41": "MOL",
    "42": "MOL",
    "43": "MOL",
    "44": "MOL",
    "45": "MOL",
    "46": "MOL",
    "47": "OPC",
    "48": "OPC",
    "49": "OPC",
    "50": "OPC",
    "51": "OPC",
    "52": "COP+NFOL",
}

leiden_str = macnair_oligo.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
macnair_oligo.obs["cell"] = mapped
macnair_oligo = macnair_oligo[macnair_oligo.obs["cell"] != "Doublet"].copy()

In [6]:
macnair_oligo.obs["celltype"] = macnair_oligo.obs["cell"].copy() #type: ignore
plot.plot_umap(macnair_oligo, MACNAIR_OLIGO, has_celltype=True)

In [16]:
plot.plot_proportions(macnair_oligo, MACNAIR_OLIGO, group_key="diagnosis", group_order=["CTR", "PPMS", "SPMS", "RRMS"], sample_key="cell", sample_order=["OPC", "COP+NFOL", "MOL", ], connect=True)
plot.plot_proportions(macnair_oligo, MACNAIR_OLIGO, sample_key="diagnosis", sample_order=["CTR", "PPMS", "SPMS", "RRMS"], group_key="cell", group_order=["OPC", "COP+NFOL", "MOL", ], connect=True)

In [21]:
plot.plot_violin(macnair_oligo, MACNAIR_OLIGO, gene=["IGSF11", "PDGFRA", "CSPG4", "BCAS1", "PDGFA", "ENPP6", "OPALIN", "MOBP"], group="cell", group_order=["OPC", "COP+NFOL", "MOL",])

In [22]:
from pipeline.utils.glmmseq import pseudobulk_glmmseq_comp
import numpy as np

deg_macnair = [
    macnair_oligo.obs["cell"] == "COP+NFOL",
    macnair_oligo.obs["cell"] == "OPC",
    macnair_oligo.obs["cell"] == "MOL",
]

choices = [
    "COPNFOL",
    "REST",
    "REST",
]

macnair_oligo.obs["deg"] = np.select(deg_macnair, choices, default="unknown")
print(macnair_oligo.obs["deg"].value_counts())

deg
REST       227234
COPNFOL       460
Name: count, dtype: int64


In [23]:
macnair_deg = pseudobulk_glmmseq_comp(macnair_oligo, "deg", group_test = "COPNFOL", group_control = "REST", random_effect_col= "donor", min_cells=10)

In [24]:
plot.plot_volcano(macnair_deg, MACNAIR_OLIGO, "COP+NFOL_vs_REST", genes=["IGSF11", "PDGFRA", "CSPG4", "BCAS1", "PDGFA", "ENPP6", "OPALIN", "MOBP"], xlim=(-7, 7), ylim=(0, 25),)

Gene outside plot limits: BCAS1 (x=3.54, y=171.83)


In [25]:
from pipeline.utils.enrichment import gsea

human_gsea = gsea(macnair_deg, species="human", series=MACNAIR_OLIGO, libraries=["GO_Biological_Process_2026", "KEGG_2026"])

In [30]:
plot.plot_gsea(human_gsea, MACNAIR_OLIGO, name="GO Biological Process", library="GO_Biological_Process_2026", 
               keywords = ["myelin", "oligodendrocyte", "differentiat" "ensheath", "sterol", "ceramide", "sulfatide", "choline", "plasmalogen", "mbp", "lipid"])

ValueError: No pathways pass the keyword, direction, and FDR filters.

In [19]:
cell_marker_genes = {
    "OPC": [ "PDGFRA", "SEZ6L", "KCNIP1", "TMEM132C", "NTNG1", "NKAIN3", "CSPG5", "SCN1A", "LAMA4", "PTPRZ1", "LHFPL3", "PCDH15", "NXPH1", ],
    "COP": ["SOX6", "UST", "BRINP1", "SH3RF3", "GPC3", "GPR17", "WASF1", "BCAS1", "EPB41L2", "ENPP6", "RAPGEF1", "EPHB1", "TCF7L2", "PRICKLE1", ],
    "NFOL": ["FRY", "GOLIM4", "DOCK6", "PIP5K1B", "ABLIM2", "RGS16", "PDGFA", "NREP", "GLRA3", "CMTM7", "TMTC1", "MAN1A1", ],
    "MOL1": ["LAMA2", "OPALIN", "SVEP1", "ROR1", "LTBP1", "DYSF", "PLXDC2", ],
    "MOL2": ["SLC5A11", "ACSBG1", "ACTN2", "HIVEP3", "ANKRD18A", "DPYSL5", "LRRC63", "MOBP", "MAG", "MBP", "PDE1A", "RNF220", "NKAIN2", ],
}
plot.plot_dotplot(val_oligo, VAL_OLIGO, cell_marker_genes, group = "leiden")

In [ ]:
leiden_to_celltype = {
    "0": "?",
    "1": "?",
    "2": "?",
    "3": "?",
    "4": "?",
    "5": "?",
    "6": "?",
    "7": "?",
    "8": "?",
    "9": "?",
    "10": "?",
    "11": "?",
    "12": "?",
    "13": "?",
    "14": "?",
    "15": "?",
    "16": "?",
    "17": "?",
    "18": "?",
    "19": "?",
    "20": "?",
    "21": "?",
    "22": "?",
    "23": "?",
    "24": "?",
    "25": "?",
    "26": "?",
    "27": "?",
    "28": "?",
    "29": "?",
    "30": "?",
    "31": "?",
    "32": "?",
    "33": "?",
    "34": "?",
    "35": "?",
    "36": "?",
    "37": "?",
    "38": "?",
    "39": "?",
    "40": "?",
    "41": "?",
    "42": "?",
    "43": "?",
    "44": "?",
    "45": "?",
    "46": "?",
    "47": "?",
    "48": "?",
    "49": "?",
    "50": "?",
    "51": "?",
    "52": "?",
    "53": "?",
    "54": "?",
    "55": "?",
    "56": "?",
    "57": "?",
}

leiden_str = val_oligo.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
val_oligo.obs["cell"] = mapped
val_oligo = val_oligo[val_oligo.obs["cell"] != "Doublet"].copy()